<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook D04: Forecasting with Transformers</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook D04: Forecasting with Transformers](../notebooks/D04_Transformers.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The windows, the model and the training loop from the notebook. `train_model` returns one extra thing
here — `curve`, the validation error after every epoch — which Exercise 2 needs.

**This notebook trains three transformers and takes around forty minutes on CPU**, most of it the long
run in Exercise 2. `FULL_RUN` behaves as it does in the notebook; every number quoted below comes from the
default workshop configuration.

In [ ]:
import sys
import importlib.util
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

sys.path.append("../notebooks")
import nb_config
import reference_scores

sns.set_theme(style="whitegrid")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    torch.set_num_threads(1)
    print(f"PyTorch {torch.__version__}")
else:
    print("PyTorch is not installed. Run 'uv sync --group dl' to follow this notebook.")

ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
    .loc["2016-01-01":"2019-12-31"]
)

LOOKBACK, HORIZON = 168, 24

FULL_RUN = False
TRAIN_STRIDE = 1 if FULL_RUN else 3
EPOCHS = 6 if FULL_RUN else 8
LEARNING_RATE = 1e-3 if FULL_RUN else 3e-3

values = load.values.astype(np.float32)
n_observations = len(values)
TEST_HOURS = VALIDATION_HOURS = 24 * 90
train_end = n_observations - TEST_HOURS - VALIDATION_HOURS

mean, std = values[:train_end].mean(), values[:train_end].std()
scaled = (values - mean) / std


def make_windows(scaled, start, stop, stride=1):
    positions = range(start, stop, stride)
    inputs = np.stack([scaled[t - LOOKBACK:t] for t in positions])
    targets = np.stack([scaled[t:t + HORIZON] for t in positions])
    return torch.tensor(inputs)[:, :, None], torch.tensor(targets)


def to_original_units(scaled_values):
    return np.asarray(scaled_values) * std + mean


def score(predictions, targets):
    return mean_absolute_error(
        to_original_units(targets).ravel(), to_original_units(predictions).ravel()
    )


class PositionalEncoding(nn.Module):
    """Add a fixed sine and cosine signature to each position."""

    def __init__(self, dimension, max_length=2000):
        super().__init__()
        encoding = torch.zeros(max_length, dimension)
        position = torch.arange(max_length).unsqueeze(1).float()
        frequency = torch.exp(
            torch.arange(0, dimension, 2).float() * (-math.log(10000.0) / dimension)
        )
        encoding[:, 0::2] = torch.sin(position * frequency)
        encoding[:, 1::2] = torch.cos(position * frequency)
        self.register_buffer("encoding", encoding)

    def forward(self, x):
        return x + self.encoding[:x.size(1)].unsqueeze(0)


class TransformerForecaster(nn.Module):
    """Encoder-only transformer with a regression head."""

    def __init__(self, d_model=32, n_heads=4, n_layers=2, feedforward=64,
                 dropout=0.1, use_positional_encoding=True, horizon=HORIZON):
        super().__init__()
        self.project = nn.Linear(1, d_model)
        self.use_positional_encoding = use_positional_encoding
        self.positional = PositionalEncoding(d_model)

        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, feedforward, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        h = self.project(x)
        if self.use_positional_encoding:
            h = self.positional(h)
        return self.head(self.encoder(h)[:, -1])


def train_model(build, epochs=None, batch_size=256, learning_rate=None, seed=0):
    """The notebook's loop, also returning the validation error at every epoch."""
    epochs = EPOCHS if epochs is None else epochs
    learning_rate = LEARNING_RATE if learning_rate is None else learning_rate

    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    model = build()
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.MSELoss()
    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, generator=generator,
    )

    started = time.time()
    best = {"validation_mae": np.inf, "epoch": 0, "weights": None}
    curve = []

    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in loader:
            optimiser.zero_grad()
            loss_function(model(batch_X), batch_y).backward()
            optimiser.step()

        model.eval()
        with torch.no_grad():
            validation_mae = score(model(X_validation).numpy(), y_validation.numpy())
        curve.append(validation_mae)

        if validation_mae < best["validation_mae"]:
            best = {"validation_mae": validation_mae, "epoch": epoch,
                    "weights": {k: v.clone() for k, v in model.state_dict().items()}}

    model.load_state_dict(best["weights"])
    model.eval()
    with torch.no_grad():
        predictions = model(X_test).numpy()

    return {
        "model": model, "test_mae": score(predictions, y_test.numpy()),
        "validation_mae": best["validation_mae"], "best_epoch": best["epoch"],
        "curve": curve, "seconds": time.time() - started,
    }


if TORCH_AVAILABLE:
    X_train, y_train = make_windows(scaled, LOOKBACK, train_end - HORIZON, TRAIN_STRIDE)
    X_validation, y_validation = make_windows(
        scaled, train_end, train_end + VALIDATION_HOURS - HORIZON
    )
    X_test, y_test = make_windows(
        scaled, train_end + VALIDATION_HOURS, n_observations - HORIZON
    )

    naive_positions = range(train_end + VALIDATION_HOURS, n_observations - HORIZON)
    naive_forecast = np.stack([values[t - 24:t - 24 + HORIZON] for t in naive_positions])
    NAIVE_MAE = mean_absolute_error(
        to_original_units(y_test.numpy()).ravel(), naive_forecast.ravel()
    )

    PREVIOUS = {
        row["Model"]: row["Test MAE"]
        for row in reference_scores.carried(
            ["LSTM (D02)", "LSTM stacked (D02)", "Simple CNN (D03)", "TCN (D03)"],
            reference_scores.AUSTRIAN_LOAD,
        )
    }

    print(f"{'FULL' if FULL_RUN else 'WORKSHOP'} run: stride {TRAIN_STRIDE}, "
          f"{EPOCHS} epochs, learning rate {LEARNING_RATE}")
    print(f"{len(X_train):,} training windows")
    print(f"Naive baseline: {NAIVE_MAE:.1f} MW")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Section 4 showed that without positional encoding the encoder cannot distinguish a week of load from the same values shuffled. Now measure what that costs: train `TransformerForecaster(use_positional_encoding=False)` and compare its test MAE against both the encoded model and the naive baseline. Given that the head reads the final position, and so always knows which value is most recent, why is the damage as large as it is?

In [ ]:
if TORCH_AVAILABLE:
    encoded = train_model(TransformerForecaster)
    unencoded = train_model(
        lambda: TransformerForecaster(use_positional_encoding=False)
    )

    position = pd.Series({
        "Transformer, with encoding": encoded["test_mae"],
        "Transformer, no encoding": unencoded["test_mae"],
        "Naive (repeat yesterday)": NAIVE_MAE,
    }).sort_values()

    print(position.round(1).to_string())
    print()
    print(f"Removing the encoding costs {unencoded['test_mae'] - encoded['test_mae']:.1f} MW, "
          f"and lands {unencoded['test_mae'] - NAIVE_MAE:.1f} MW *worse* than the baseline")

**It is not a degradation, it is a collapse: 933.6 MW against 326.7 with the encoding, and against
563.3 for a one-line baseline that repeats yesterday.**

Removing thirty-two numbers added to each position turns a model that beats the naive forecast by 42% into
one that is 66% *worse* than it. Nothing else in Part D fails this badly — even the plain RNN in Notebook
[D02](../notebooks/D02_Recurrent_networks.ipynb), the weakest architecture in the course, stayed under the
baseline.

To the question of why the head does not save it. It is true that reading the final position gives the
model one piece of order information for free: it always knows which value is the most recent, because
that is the one it is reading. The trouble is that this is *all* it knows.

Follow what the final position's representation is actually built from. Self-attention computes it as a
weighted sum over every position in the window, and section 4 established that without an encoding the
encoder is permutation-equivariant: **the weights depend on the values being attended to, not on where
they sit.** So the representation the head receives is, in effect,

> the most recent value, plus a value-weighted summary of the *set* of the previous 167 hours.

A set. The model can tell that the window contained a reading of 5,800 MW and one of 9,100 MW. It cannot
tell whether the 9,100 was yesterday evening or six days ago.

For this series that is close to fatal, because **the whole forecastable structure of electricity load is
positional**. Consider what a day-ahead forecast needs:

- Which hour of the day the window ends on, so it knows where in the daily cycle the next 24 hours begin.
  The daily cycle is the dominant signal, and its phase is purely a matter of position.
- Whether the last few days were weekdays or a weekend, which is position modulo 168.
- Whether load has been rising or falling across the window, which is the *order* of the values, not their
  multiset.

Every one of those is destroyed by shuffling, and the model without an encoding sees exactly what a
shuffled window would give it. What it retains — the level, the spread, the most recent value — supports
about as good a forecast as "tomorrow will average what this week averaged", and that is roughly the score
it gets.

**The general lesson is about where a model's inductive bias comes from, and it is the thread running
through all of Part D.** A convolution cannot ignore order: locality is built into the operation. A
recurrent network cannot ignore order: it processes steps in sequence. Attention has no such commitment —
it is the most general of the three, and generality here means *nothing is assumed, so everything must be
supplied*. The positional encoding is not a refinement bolted onto the architecture; it is the mechanism
by which a transformer is told it is looking at a sequence at all.

Which is also the practical warning. This failure produces no error and no warning. The model trains, the
loss falls, the shapes are right, and the output is worse than a baseline. If you build an attention model
and it underperforms surprisingly, check that position is reaching it before you change anything else.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> The conclusion above claims the transformer is under-trained, on the evidence that validation error was still falling at the final epoch. Test it: record the validation error at every epoch, then retrain for three times as many and see whether the curve flattens and where the test score ends up. Does more training close the gap to the convolutional models?

In [ ]:
if TORCH_AVAILABLE:
    long_run = train_model(TransformerForecaster, epochs=EPOCHS * 3)

    training = pd.DataFrame({
        "epochs": [EPOCHS, EPOCHS * 3],
        "test MAE": [encoded["test_mae"], long_run["test_mae"]],
        "best validation MAE": [encoded["validation_mae"], long_run["validation_mae"]],
        "best epoch": [encoded["best_epoch"] + 1, long_run["best_epoch"] + 1],
        "seconds": [encoded["seconds"], long_run["seconds"]],
    }).set_index("epochs")

    print(training.round(1).to_string())
    print()
    print("Where that lands it:")
    print(pd.Series({**PREVIOUS,
                     "Transformer, 8 epochs": encoded["test_mae"],
                     "Transformer, 24 epochs": long_run["test_mae"]})
          .sort_values().round(1).to_string())

In [ ]:
if TORCH_AVAILABLE:
    fig, ax = plt.subplots(figsize=(12, 5))

    epochs = np.arange(1, len(long_run["curve"]) + 1)
    ax.plot(epochs, long_run["curve"], marker="o", markersize=4, linewidth=1.5,
            color="mediumpurple", label="validation MAE")
    ax.axvline(EPOCHS, color="crimson", linestyle="--", linewidth=1.2)
    ax.text(EPOCHS + 0.3, max(long_run["curve"]) * 0.8,
            "the notebook stops here", fontsize=9, color="crimson")
    ax.scatter([long_run["best_epoch"] + 1], [long_run["validation_mae"]],
               color="crimson", s=80, zorder=3, label="best epoch")

    for mae, label, colour in [(PREVIOUS["Simple CNN (D03)"], "Simple CNN", "steelblue"),
                               (PREVIOUS["TCN (D03)"], "TCN", "seagreen")]:
        ax.axhline(mae, color=colour, linestyle=":", linewidth=1.2)
        ax.text(len(epochs) * 0.82, mae + 5, label, fontsize=9, color=colour)

    ax.set_title("Validation error over 24 epochs", fontsize=13, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation MAE (MW)")
    ax.set_ylim(150, 550)
    ax.legend()
    ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

**The diagnosis was right. Tripling the training takes the test score from 326.7 to 275.9, an
improvement of 15.6%, and the model overtakes the Simple CNN.**

The curve settles the first half of the question. Over the notebook's eight epochs it falls monotonically
— 829, 501, 315, 277, 253, 248, 234, 225 — with no sign of a floor, which is the observation the
conclusion was based on. Run it three times as long and it does eventually flatten: the best epoch is 23
of 24, but the last sixteen epochs together buy less than the first eight did on their own, and from about
epoch 11 the curve is mostly noise between 185 and 210.

So the ranking in section 7 changes, but only partly:

| | test MAE |
|---|---|
| TCN (D03) | 260.6 |
| **Transformer, 24 epochs** | **275.9** |
| Simple CNN (D03) | 291.6 |
| Transformer, 8 epochs | 326.7 |
| LSTM stacked (D02) | 350.4 |

**It passes the two-layer CNN and does not catch the TCN.** The notebook's summary — "better than both
LSTMs, worse than both CNNs" — is a statement about a particular training budget rather than about the
architectures, and half of it dissolves when the budget changes.

The cost is the part that does not dissolve. The long run took **close to 1,500 seconds against the Simple
CNN's 22**, a factor of sixty-six for a model that is 5% better, and it still loses to a TCN that
trains in about 280. On this problem the transformer is not merely worse value; it is worse value by two
orders of magnitude in training time.

One caveat on comparability, and it cuts against the transformer rather than for it. The CNN figures come
from Notebook [D03](../notebooks/D03_Convolutional_networks.ipynb), which has no workshop mode and trains
on every window, while this transformer trains on every third. At 24 epochs of a third of the data the
long run does roughly a third more gradient work than the notebook's own full run, and it beats that full
run's 304.2 — so the 275.9 is not a stride artefact. But the comparison is not clean, and the honest way
to put the conclusion is the one section 8 already reaches: **the conditions under which a transformer
wins are many series and a lot of data, and one Austrian load series is neither.** More training moves it
up one place in a ranking it should not have been expected to top.

---

Back to [Notebook D04](../notebooks/D04_Transformers.ipynb), or on to
[Notebook D05](../notebooks/D05_Specialised_architectures.ipynb).